In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random
import plot_training


# Environment class - this is the gridworld environment that the agents will interact with
class Environment:
    def __init__(self, grid_size=5, num_agents=1):
        self.grid_size = grid_size
        self.num_agents = num_agents
        self.num_targets = num_agents
        self.agents = []
        self.targets = []
        self.reset()

    # method - reset() - resets the environment to a random state and returns the initial state
    def reset(self):
        # resets grid and agents
        self.agents = []
        self.targets = []
        # place agents randomly
        for _ in range(self.num_agents):
            x = random.randint(0, self.grid_size - 1)
            y = random.randint(0, self.grid_size - 1)
            self.agents.append((x, y))
        # place targets randomly
        for _ in range(self.num_targets):
            x = random.randint(0, self.grid_size - 1)
            y = random.randint(0, self.grid_size - 1)
            # make sure target isn't on top of an agent
            while (x, y) in self.agents:
                x = random.randint(0, self.grid_size - 1)
                y = random.randint(0, self.grid_size - 1)
            self.targets.append((x, y))

    # method - step(action) - takes an action and returns the new_state, reward, done
    def step(self, actions):
        # actions can be 0: up, 1: down, 2: left, 3: right, 4: wait
        new_agents = []
        reward = 1
        done = True
        # move agents (watch for out of bounds)
        print("input actions: ", actions)
        for i, action in enumerate(actions):
            print("executing action: ", action, "for agemt: ", i)
            x, y = self.agents[i]
            if action == 0 and y > 0:  # up
                y -= 1
            elif action == 1 and y < self.grid_size - 1:  # down
                y += 1
            elif action == 2 and x > 0:  # left
                x -= 1
            elif action == 3 and x < self.grid_size - 1:  # right
                x += 1
            new_agents.append((x, y))
        self.agents = new_agents
        # if both agents are on targets then reward = 1 done = True
        for (x, y) in self.agents:
            if (x,y) not in self.targets:
                reward = 0
                done = False
        return reward, done

    # method - state() - returns the current state of the environment
    # format (xA1, yA1, xA2, yA2, ..., xT1, yT1, xT2, yT2, ...)
    def state(self, agent_index):
        state = []
        # add own position first
        state.extend([self.agents[agent_index][0] / (self.grid_size - 1.0), self.agents[agent_index][1] / (self.grid_size - 1.0)])
        for agent in self.agents:
            if agent != self.agents[agent_index]:
                state.extend(
                    [agent[0] / (self.grid_size - 1.0), agent[1] / (self.grid_size - 1.0)]
                )  # normalize to [0, 1]
        for target in self.targets:
            state.extend(
                [target[0] / (self.grid_size - 1.0), target[1] / (self.grid_size - 1.0)]
            )  # normalize to [0, 1]
        return torch.tensor(state, dtype=torch.float32)

    # method - render() - prints the current state of the environment to the console
    def render(self):
        grid = [["." for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        for x, y in self.targets:
            grid[y][x] = "T"
        for i, (x, y) in enumerate(self.agents):
            grid[y][x] = f"{i}"
        for row in grid:
            print(" ".join(row))
        print()

    def set(self, agents, targets):
        self.agents = agents
        self.targets = targets


In [31]:
testEnv = Environment(5, 2)
testEnv.reset()
testEnv.render()
print(testEnv.state(0))
print(testEnv.state(1))
reward, done = testEnv.step([4,2])
print(testEnv.state(0))
print(testEnv.state(1))


. . 1 . .
. . . . 0
. T . T .
. . . . .
. . . . .

tensor([1.0000, 0.2500, 0.5000, 0.0000, 0.2500, 0.5000, 0.7500, 0.5000])
tensor([0.5000, 0.0000, 1.0000, 0.2500, 0.2500, 0.5000, 0.7500, 0.5000])
executing action:  4 for agemt:  0
executing action:  2 for agemt:  1
tensor([1.0000, 0.2500, 0.2500, 0.0000, 0.2500, 0.5000, 0.7500, 0.5000])
tensor([0.2500, 0.0000, 1.0000, 0.2500, 0.2500, 0.5000, 0.7500, 0.5000])


In [19]:
reward, done = testEnv.step([4,2])
print(reward, " ", done)
testEnv.render()
print(testEnv.state(0))
print(testEnv.state(1))

1   True
. . . . .
1 . . . 0
. . . . .
. . . . .
. . . . .

tensor([1.0000, 0.2500, 0.0000, 0.2500, 1.0000, 0.2500, 0.0000, 0.2500])
tensor([0.0000, 0.2500, 1.0000, 0.2500, 1.0000, 0.2500, 0.0000, 0.2500])


In [20]:
class DQN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [21]:
layer_size = 4
agents = 2

networks = []
for i in range(agents):
    q_network = DQN(input_size=4 * agents, hidden_size=layer_size, output_size=4)
    target_network = DQN(input_size=4 * agents, hidden_size=layer_size, output_size=4)
    target_network.load_state_dict(q_network.state_dict())
    networks.append([q_network,target_network])

print(networks)

[[DQN(
  (fc1): Linear(in_features=8, out_features=4, bias=True)
  (fc2): Linear(in_features=4, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=4, bias=True)
), DQN(
  (fc1): Linear(in_features=8, out_features=4, bias=True)
  (fc2): Linear(in_features=4, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=4, bias=True)
)], [DQN(
  (fc1): Linear(in_features=8, out_features=4, bias=True)
  (fc2): Linear(in_features=4, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=4, bias=True)
), DQN(
  (fc1): Linear(in_features=8, out_features=4, bias=True)
  (fc2): Linear(in_features=4, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=4, bias=True)
)]]


In [28]:
rows = 2
cols = 0

# Initializes a 3x4 grid filled with 0s
list = [[0 for _ in range(0)] for _ in range(2)]
print(list)
list[0].append(([1,2,3,4], 1, 0, [1,2,3,4], False))
list[1].append(([1,2,3,4], 1, 0, [1,2,3,4], False))
print(list)

[[], []]
[[([1, 2, 3, 4], 1, 0, [1, 2, 3, 4], False)], [([1, 2, 3, 4], 1, 0, [1, 2, 3, 4], False)]]


In [40]:
losses = [1,2,3,4]
#result = [(x + y) / 2.0 for x, y in zip(losses[0], losses[1])]

print(sum(losses))


10
